In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import ttk

In [ ]:
df = pd.read_csv('Data Set SMK - Data Fix.csv')
a = pd.read_excel('Data Set SMK.xlsx')

In [ ]:
df.info()

In [ ]:
df.head(80)

In [ ]:
df['Kode Rumah'].value_counts()

In [ ]:
djalan = [
    ["A1", "C4", 400],
    ["A1", "C13", 300],
    ["A1", "C7", 400],
    ["A1", "B2", 400],
    ["C7", "C4", 400],
    ["C7", "C1", 300],
    ["C1", "C2", 400],
    ["C1", "C6", 400],
    ["C2", "C8", 200],
    ["C8", "C6", 200],
    ["C8", "C14", 400],
    ["C14", "B1", 200],
    ["B1", "C9", 200],
    ["C9", "C3", 200],
    ["C3", "B2", 400],
    ["C3", "C10", 400],
    ["B2", "C5", 200],
    ["C5", "C6", 100],
    ["C5", "B1", 200],
    ["C6", "C7", 300],
    ["C6", "C1", 400],
    ["C6", "C5", 100],
    ["C6", "B2", 400],
    ["C10", "C12", 300],
    ["C12", "C13", 300],
    ["C13", "C4", 300],
    ["C4", "C7", 400]
]

waktu_value = {
    100:10,
    200:15,
    300:20,
    400:25
}

df_jalan = pd.DataFrame(djalan, columns=['Asal', 'Tujuan', 'Jarak'])
df_jalan['Waktu'] = df_jalan['Jarak'].map(waktu_value)

In [ ]:
df_jalan.head(75)

In [ ]:
def djikstra(titik_awal, tujuan, df_jalan, df_data_rumah, based='Jarak'):
    semua_titik = pd.unique(df_jalan[['Asal', 'Tujuan']].values.ravel())

    jarak = pd.Series(data=[np.inf] * len(semua_titik), index=semua_titik)
    jarak[titik_awal] = 0
    prev = pd.Series(data=[None] * len(semua_titik), index=semua_titik)

    queue = list(semua_titik)

    while queue:
        now = jarak[queue].idxmin()
        queue.remove(now)

        if now == tujuan:
            break


        kondisi = (df_jalan['Asal'] == now) | (df_jalan['Tujuan'] == now)
        for _, baris in df_jalan[kondisi].iterrows():
            tetangga = baris['Tujuan'] if baris['Asal'] == now else baris['Asal']
            baru = jarak[now] + baris[based]
            if baru < jarak[tetangga]:
                jarak[tetangga] = baru
                prev[tetangga] = now


        # tetangga_asal = df_jalan[df_jalan['Asal'] == now]
        # for _, baris in tetangga_asal.iterrows():
        #     baru = jarak[now] + baris['Jarak']
        #     if baru < jarak[baris['Tujuan']]:
        #         jarak[baris['Tujuan']] = baru
        #         prev[baris['Tujuan']] = now

        # tetangga_tujuan = df_jalan[df_jalan['Tujuan'] == now]
        # for _, baris in tetangga_tujuan.iterrows():
        #     baru = jarak[now] + baris['Jarak']
        #     if baru < jarak[baris['Asal']]:
        #         jarak[baris['Asal']] = baru
        #         prev[baris['Asal']] = now
        
    rute = []
    curr = tujuan
    while curr is not None:
        rute.append(curr)
        curr = prev[curr]
    rute = rute[::-1]

    total_j = 0
    total_w_jalan = 0
    total_w_kunjung = 0
    
    for i in range(len(rute)-1):
        s, e = rute[i], rute[i+1]
        
        # 1. Hitung Waktu & Jarak Perjalanan
        mask = ((df_jalan['Asal'] == s) & (df_jalan['Tujuan'] == e)) | \
               ((df_jalan['Asal'] == e) & (df_jalan['Tujuan'] == s))
        data_jalur = df_jalan[mask].iloc[0]
        total_j += data_jalur['Jarak']
        total_w_jalan += data_jalur['Waktu']
        
        # 2. Hitung Waktu Kunjungan di titik 'e' (Tujuan)
        # Ambil luas rumah dari df (Data Set SMK) berdasarkan Kode Rumah
    luas = df_data_rumah[df_data_rumah['Kode Rumah'] == e]['Luas Rumah'].values[0]
    
    # Logika: > 200m -> 30 mnt, <= 200m -> 15 mnt
    if luas > 200:
        total_w_kunjung += 30 
    else:
        total_w_kunjung += 15

    total_waktu_akhir = total_w_jalan + total_w_kunjung
    
    return rute, total_j, total_w_jalan, total_w_kunjung, total_waktu_akhir

In [ ]:
def kode(nama_target, df):
    hasil = df[df['Nama'] == nama_target]['Kode Rumah'].values[0]
    return hasil

In [ ]:
df['Nama'].unique()

In [ ]:
# df adalah DataFrame dari 'Data Set SMK - Data Fix.csv'
asal_wong = kode('Pak Sutajo', df)
tujuan_wong = kode('Pak Bakri ', df)
rute, jarak, w_jalan, w_kunjung, total = djikstra(asal_wong, tujuan_wong, df_jalan, df)

print(f"Rute: {' -> '.join(rute)}")
print(f"Total Jarak: {jarak} meter")
print(f"Waktu di Perjalanan: {w_jalan} menit")
print(f"Waktu Kunjungan: {w_kunjung} menit")
print(f"Estimasi Total (Sampai Selesai): {total} menit")

In [ ]:
# # Cari kode rumah berdasarkan nama
# asal_nama = "Pak Sutajo"
# tujuan_nama = "Pak Hari"

# kode_asal = kode(asal_nama, df)
# kode_tujuan = kode(tujuan_nama, df)

# # Jalankan algoritma Dijkstra
# rute, jarak_total, taim = djikstra(kode_asal, kode_tujuan, df_jalan, df)

# print(f"Rute terpendek dari {asal_nama} ({kode_asal}) ke {tujuan_nama} ({kode_tujuan}):")
# print(" -> ".join(rute))
# print(f"Total Jarak: {jarak_total} meter")
# print(taim)

In [ ]:

def hitung_rute_gui():
    nama_a = combo_asal.get()
    nama_t = combo_tujuan.get()
    
    k_asal = kode(nama_a, df)
    k_tujuan = kode(nama_t, df)
    rute, jarak, w_jalan, w_kunjung, total = djikstra(k_asal, k_tujuan, df_jalan, df)
    
    label_hasil.config(text=f"Rute: {' -> '.join(rute)}\nJarak: {jarak}m\nWaktu Perjalanan {w_jalan}\nWaktu Kunjungan {w_kunjung}\nTotal Waktu {total}")

root = tk.Tk()

title_atas = tk.Label(root, text='Maps AI (Aseli Ini)', font=('Arial', 28, 'bold'))
title_atas.pack(pady=10)

title_asal = tk.Label(root, text='Pilih Lokasi Asal Anda').pack()
combo_asal = ttk.Combobox(root, values=list(df['Nama'].unique()))
combo_asal.pack(pady=10)

title_tujuan = tk.Label(root, text='Pilih Lokasi Tujuan Anda').pack()
combo_tujuan = ttk.Combobox(root, values=list(df['Nama'].unique()))
combo_tujuan.pack()

btn = tk.Button(root, text="Cari Jalan!", command=hitung_rute_gui, font=('Arial', 11))
btn.pack(pady=10)

label_hasil = tk.Label(root)
label_hasil.pack()

root.mainloop()

In [ ]:
df.head(10)

## amodel 